<a href="https://colab.research.google.com/github/maneeha/KGLLM/blob/main/PubMedDCM_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install datasets pandas nltk


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 13.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 whic

In [3]:
# Load dataset from Hugging Face
from datasets import load_dataset

# For data manipulation and visualization
import pandas as pd

# For text processing and filtering
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Download NLTK resources (only needed once)
nltk.download("punkt")
nltk.download("stopwords")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
!pip install torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 79.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [12]:
from datasets import load_dataset

# Load dataset
dataset = load_dataset("qiaojin/PubMedQA" , "pqa_artificial")
train_data = dataset["train"]  # Select the 'train' split

# Define relevant keywords
keywords = ["diabetes control", "blood sugar", "glucose management", "insulin therapy",
            "HbA1c", "diabetes treatment", "diabetes management"]

# Function to filter dataset based on keywords
def filter_diabetes_control(example):
    question_text = example["question"] if isinstance(example["question"], str) else example["question"].get("text", "")
    context_text = example["context"] if isinstance(example["context"], str) else str(example["context"])  # Convert dict to string if needed
    long_answer_text = example.get("long_answer", "")  # Handle missing key
    final_decision_text = example.get("final_decision", "")  # Handle missing key

    combined_text = (question_text + " " + context_text + " " + long_answer_text + " " + final_decision_text).lower()
    return any(keyword in combined_text for keyword in keywords)

# Apply filtering
filtered_data = train_data.filter(filter_diabetes_control)

# ✅ Convert to list for easier indexing
filtered_list = [dict(row) for row in filtered_data]

# Display results
print(f"Total relevant rows found: {len(filtered_list)}")
for i in range(min(5, len(filtered_list))):
    print(f"Question: {filtered_list[i]['question']}")
    print(f"Context: {filtered_list[i]['context']}")  # May still contain nested data
    print(f"Long Answer: {filtered_list[i].get('long_answer', 'N/A')}")
    print(f"Final Decision: {filtered_list[i].get('final_decision', 'N/A')}")
    print("-" * 80)


Filter:   0%|          | 0/211269 [00:00<?, ? examples/s]

Total relevant rows found: 457
Question: Does low perfusion index affect the difference in glucose level between capillary and venous blood?
Context: {'contexts': ['In emergency cases, finger stick testing is primarily used to check the blood glucose value of patients since it takes longer to obtain the venous value. In critical patients, under conditions that cause an increase in metabolic state and level of stress, there occurs considerable difference in glucose levels between capillary and venous measurements. This study aimed to investigate the comparability of capillary and venous glucose values, according to the perfusion index level obtained with the Masimo Radical-7(®) device, in critical patients aged 18 years and over.', 'We conducted this prospective and observational study in the emergency department of the Eskisehir Osmangazi University hospital between November 3, 2008 and February 2, 2009.', 'The blood glucose of 300 critical patients was checked by finger stick in the e

In [13]:
!pip install pandas openpyxl


In [14]:
import pandas as pd

# Convert filtered dataset to a Pandas DataFrame
df = pd.DataFrame(filtered_list)

# Save to an Excel file
df.to_excel("filtered_pubmedqa.xlsx", index=False, engine="openpyxl")

print("✅ Data successfully saved as 'filtered_pubmedqa.xlsx'")


✅ Data successfully saved as 'filtered_pubmedqa.xlsx'


In [15]:
from google.colab import files
files.download("filtered_pubmedqa.xlsx")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Just for Diabebtes Control and Diabetes Management

In [16]:
from datasets import load_dataset

# Load dataset
dataset = load_dataset("qiaojin/PubMedQA" , "pqa_artificial")
train_data = dataset["train"]  # Select the 'train' split

# Define relevant keywords
keywords = ["diabetes control", "diabetes management"]

# Function to filter dataset based on keywords
def filter_diabetes_control(example):
    question_text = example["question"] if isinstance(example["question"], str) else example["question"].get("text", "")
    context_text = example["context"] if isinstance(example["context"], str) else str(example["context"])  # Convert dict to string if needed
    long_answer_text = example.get("long_answer", "")  # Handle missing key
    final_decision_text = example.get("final_decision", "")  # Handle missing key

    combined_text = (question_text + " " + context_text + " " + long_answer_text + " " + final_decision_text).lower()
    return any(keyword in combined_text for keyword in keywords)

# Apply filtering
filtered_data = train_data.filter(filter_diabetes_control)

# ✅ Convert to list for easier indexing
filtered_list = [dict(row) for row in filtered_data]

# Display results
print(f"Total relevant rows found: {len(filtered_list)}")
for i in range(min(5, len(filtered_list))):
    print(f"Question: {filtered_list[i]['question']}")
    print(f"Context: {filtered_list[i]['context']}")  # May still contain nested data
    print(f"Long Answer: {filtered_list[i].get('long_answer', 'N/A')}")
    print(f"Final Decision: {filtered_list[i].get('final_decision', 'N/A')}")
    print("-" * 80)

Filter:   0%|          | 0/211269 [00:00<?, ? examples/s]

Total relevant rows found: 117
Question: Does exome sequencing identify novel ApoB loss-of-function mutations causing hypobetalipoproteinemia in type 1 diabetes?
Context: {'contexts': ['Diabetic patients commonly suffer from disturbances in production and clearance of plasma lipoproteins, known as diabetic dyslipidemia, resulting in an increased risk of coronary heart disease. The study aimed to examine the cause of hypobetalipoproteinemia in two patients with type 1 diabetes.', 'The Diabetes Control and Complications Trial (DCCT) is a study demonstrating that intensive blood glucose control delays the onset and progression of type 1 diabetes complications. Hypobetalipoproteinemia was present in two DCCT subjects, IDs 1427 and 1078, whose LDL-C levels were 36 and 28 mg/dL, respectively, and triglyceride levels were 20 and 28 mg/dL, respectively. We performed exome sequencing on genomic DNA from the two patients with hypobetalipoproteinemia.', 'The subjects 1427 and 1078 had heterozygou

In [17]:
import pandas as pd

# Convert filtered dataset to a Pandas DataFrame
df = pd.DataFrame(filtered_list)

# Save to an Excel file
df.to_excel("filtered_pubmedqa_dcm.xlsx", index=False, engine="openpyxl")

print("✅ Data successfully saved as 'filtered_pubmedqa_dcm.xlsx'")


✅ Data successfully saved as 'filtered_pubmedqa.xlsx'


In [18]:
from google.colab import files
files.download("filtered_pubmedqa_dcm.xlsx")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>